# Task 2 — Resolution scaling and crop-then-click

**Owner:** Dong Yiwei  
**Hypothesis (H2):** Higher effective resolution and a two-stage crop-then-click policy will improve ScreenSpot-Pro most, especially on small targets — recovering 5-10 pp for generalists.

**Variations (5 cells)**
1. Single-stage at 0.5× native
2. Single-stage at 1× native (proposal default)
3. Single-stage at 2× native
4. Crop-then-click, 512×512 window
5. Crop-then-click, 768×768 window

**Deeper analysis:** failure analysis on small icons + ablation of resolution-only vs. crop-only gains.

In [ ]:
from ais5.data import load_benchmark
from ais5.eval import evaluate_model
from ais5.eval.breakdown import by_target_size
from ais5.models import get_model
from ais5.tile import CropConfig, crop_then_click, scale_image
from ais5.utils import set_global_seed, setup_logging

setup_logging()
set_global_seed(42)

# Use the best LoRA adapter from Task 1 if available.
model = get_model('qwen2.5-vl-3b')  # peft_adapter='checkpoints/qwen-r32-50k' once trained

## Resolution sweep

In [ ]:
import dataclasses, copy

results = {}
for factor in [0.5, 1.0, 2.0]:
    samples = []
    for s in load_benchmark('screenspot-pro'):
        scaled = copy.copy(s)
        scaled.image = scale_image(s.image, factor)
        scaled.image_size = scaled.image.size
        # bbox is in original pixels — rescale it to match.
        x1, y1, x2, y2 = s.bbox
        scaled.bbox = (x1 * factor, y1 * factor, x2 * factor, y2 * factor)
        samples.append(scaled)
    run = evaluate_model(model, samples, benchmark='screenspot-pro', limit=100)
    results[f'{factor}x'] = run.accuracy
    print(f'{factor}× native: acc={run.accuracy:.4f}')

## Crop-then-click sweep

Wraps the model's predict() in `crop_then_click(...)` to produce a refined click. We measure end-to-end accuracy *and* per-step latency (since refinement doubles the inference count).

In [ ]:
import time

from ais5.eval.click import ClickResult, point_in_bbox

def crop_eval(model, samples, crop_size, limit=100):
    cfg = CropConfig(crop_size=crop_size)
    rs = []
    for s in samples[:limit]:
        t0 = time.perf_counter()
        out = crop_then_click(model, s.image, s.instruction, cfg=cfg)
        dt_ms = (time.perf_counter() - t0) * 1000.0
        pred = out.parsed.point
        correct = bool(pred is not None and point_in_bbox(pred, s.bbox))
        rs.append(ClickResult(
            sample_id=s.sample_id or '',
            pred=pred,
            bbox=s.bbox,
            correct=correct,
            benchmark=s.benchmark,
            target_relative_area=s.target_relative_area,
            ui_type=s.ui_type,
            target_type=s.target_type,
            raw_response=out.text,
            latency_ms=dt_ms,
        ))
    acc = sum(r.correct for r in rs) / len(rs) if rs else 0.0
    return rs, acc

samples_pro = list(load_benchmark('screenspot-pro'))
for crop in [512, 768]:
    rs, acc = crop_eval(model, samples_pro, crop_size=crop)
    print(f'crop={crop}px: acc={acc:.4f}')

## Failure analysis on small icons

Group results by target-area bucket. The hypothesis is that the `xs` and `s` buckets benefit most from crop-then-click.

In [ ]:
by_target_size(rs)